# Homework Part A - Sessions 1, 2 and 3

**Student:** Daniela Gonzalez  
**Course:** Computer Vision and Speech Recognition - EADA  
**Dataset:** MILK10k

In [1]:
# --- Setup ------------------------------------------------------------------
import os
import sys
from pathlib import Path
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Make the local package importable when the notebook lives in notebooks/
ROOT = Path.cwd().resolve()
if (ROOT / "notebooks").is_dir():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "notebooks").is_dir():
    sys.path.insert(0, str(ROOT.parent))

from milk10k_pipeline import config

config.sanity_check()

df = pd.read_csv(config.METADATA_PATH)
gt = pd.read_csv(config.GT_PATH)

print()
print("metadata:", df.shape, "| gt:", gt.shape)
df.head(3)

ROOT             : /Users/danielagonzalez/Downloads/milk10k-eda
METADATA_PATH    : /Users/danielagonzalez/Downloads/milk10k-eda/data/raw/milk10k/metadata.csv exists: True
GT_PATH          : /Users/danielagonzalez/Downloads/milk10k-eda/data/raw/milk10k/supplements/training_gt.csv exists: True
IMG_DIR          : /Users/danielagonzalez/Downloads/milk10k-eda/data/raw/milk10k/images exists: True
FIGURES_DIR      : /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_task3
SPLITS_DIR       : /Users/danielagonzalez/Downloads/milk10k-eda/splits
SEED             : 0
IMAGE_SIZE       : (224, 224)

metadata: (10480, 17) | gt: (5240, 12)


,isic_id,attribution,copyright_license,age_approx,anatom_site_general,anatom_site_special,concomitant_biopsy,diagnosis_1,diagnosis_2,diagnosis_3,diagnosis_4,diagnosis_confirm_type,image_manipulation,image_type,lesion_id,melanocytic,sex
0,ISIC_0051817,MILK study team,CC-BY-NC,70.0,upper extremity,NaN,True,Malignant,Malignant epidermal proliferations,"Squamous cell carcinoma, Invasive",NaN,histopathology,instrument only,dermoscopic,IL_1612768,NaN,male
1,ISIC_0073863,MILK study team,CC-BY-NC,5.0,NaN,NaN,True,Benign,Benign melanocytic proliferations,Nevus,"Nevus, Reed",histopathology,instrument only,dermoscopic,IL_2547802,True,female
2,ISIC_0075884,MILK study team,CC-BY-NC,10.0,upper extremity,NaN,True,Benign,Benign melanocytic proliferations,Nevus,"Nevus, Acral",histopathology,instrument only,clinical: close-up,IL_9270970,True,female


## A1.1 Cross-check the two label files

In [2]:
# --- A1.1 Cross-check the two label files -----------------------------------

# (a) the set of lesion_id is identical in both files
assert set(df["lesion_id"]) == set(gt["lesion_id"]), "lesion_id sets differ"
print("(a) OK: same lesion_id set,", len(set(df["lesion_id"])), "lesions")

# (b) every lesion has exactly ONE positive class in the one-hot gt
class_cols = [c for c in gt.columns if c != "lesion_id"]
pos_per_row = gt[class_cols].sum(axis=1)
assert (pos_per_row == 1).all(), "some lesions do not have exactly one positive class"
print("(b) OK: exactly one positive class per lesion")

# (c) does each of the 11 classes map to exactly ONE diagnosis_1?
merged = df.merge(gt, on="lesion_id", how="inner")

long = merged.melt(
    id_vars=["lesion_id", "isic_id", "diagnosis_1"],
    value_vars=class_cols,
    var_name="dx_class",
    value_name="is_dx",
)
long = long[long["is_dx"] == 1]

mapping = long.groupby("dx_class")["diagnosis_1"].agg(lambda s: sorted(s.unique()))
mapping_df = pd.DataFrame({
    "n_dx1_values": mapping.apply(len),
    "diagnosis_1_values": mapping.apply(lambda x: ", ".join(x)),
})
print()
print("(c) mapping dx_class -> diagnosis_1:")
print(mapping_df)

violations = mapping_df[mapping_df["n_dx1_values"] > 1]
print()
if len(violations) == 0:
    print("(c) no violations: every class maps to exactly one diagnosis_1")
else:
    print("(c) violations:")
    print(violations)
    for cls in violations.index:
        sub = long[long["dx_class"] == cls]
        print()
        print(f"  {cls}: distribution across diagnosis_1")
        print(sub["diagnosis_1"].value_counts())

# (d) each diagnosis_3 belongs to exactly one diagnosis_2;
#     each diagnosis_2 belongs to exactly one diagnosis_1
d23 = df.dropna(subset=["diagnosis_3"]).groupby("diagnosis_3")["diagnosis_2"].nunique()
d21 = df.dropna(subset=["diagnosis_2"]).groupby("diagnosis_2")["diagnosis_1"].nunique()
print()
print("(d) diagnosis_3 values mapping to >1 diagnosis_2:", int((d23 > 1).sum()))
print("(d) diagnosis_2 values mapping to >1 diagnosis_1:", int((d21 > 1).sum()))

# (e) the two images of a lesion have identical age, sex, site and diagnosis fields
fields = ["age_approx", "sex", "anatom_site_general", "diagnosis_1",
          "diagnosis_2", "diagnosis_3", "diagnosis_4"]
inconsistent = df.groupby("lesion_id")[fields].nunique()
violations_e = (inconsistent > 1).any(axis=1)
print()
print("(e) lesions with inconsistent fields between their 2 images:",
      int(violations_e.sum()))

(a) OK: same lesion_id set, 5240 lesions
(b) OK: exactly one positive class per lesion

(c) mapping dx_class -> diagnosis_1:
          n_dx1_values        diagnosis_1_values
dx_class                                        
AKIEC                2  Indeterminate, Malignant
BCC                  1                 Malignant
BEN_OTH              1                    Benign
BKL                  1                    Benign
DF                   1                    Benign
INF                  1                    Benign
MAL_OTH              1                 Malignant
MEL                  1                 Malignant
NV                   1                    Benign
SCCKA                1                 Malignant
VASC                 1                    Benign

(c) violations:
          n_dx1_values        diagnosis_1_values
dx_class                                        
AKIEC                2  Indeterminate, Malignant

  AKIEC: distribution across diagnosis_1
diagnosis_1
Malignant        360

### A1.1 Written answer

**(c) What was found.** The mapping from the 11-class scheme to `diagnosis_1` is almost one-to-one: 10 of the 11 classes map cleanly to a single `diagnosis_1` label. The only exception is `AKIEC`, which contains 360 Malignant lesions and 246 Indeterminate lesions. Every other class (BCC, MEL, SCCKA, MAL_OTH -> Malignant; NV, BKL, DF, INF, VASC, BEN_OTH -> Benign) is unambiguous.

**Implication for the label strategy.** Because `AKIEC` splits across two `diagnosis_1` values, `diagnosis_1` cannot be fully derived from the 11-class label without extra information. Two options follow: (i) treat `diagnosis_1` as the primary task and the 11-class scheme as a higher-resolution variant that only makes sense for the 10 clean classes; or (ii) derive `diagnosis_1` from the 11 classes plus one binary flag for Indeterminate `AKIEC` lesions. Both are defensible; option (i) is simpler and is what will be used in the pipeline.

**Which checks to repeat on any new medical dataset.** At least: (a) the identifier sets must match across files that describe the same entity; (b) one-hot labels must have exactly one positive class per row; (c) the class hierarchy must be a tree, i.e. each child class must have exactly one parent; (e) duplicate rows for the same entity must agree on demographic and diagnosis fields. Checks (a) and (c) are the ones most likely to catch mistakes introduced upstream.

**Which failure is most dangerous.** The `AKIEC` case is the most dangerous because it silently breaks the assumption that the 11-class scheme is a strict refinement of `diagnosis_1`. If that assumption were baked into the pipeline without checking, training and evaluation could mix two incompatible label definitions, and any derived metric would be wrong in ways that are hard to notice.


## A1.2 Ask the data five questions

Each question is answered with a few lines of pandas. All results are on the
lesion-level table when the question is about lesions, and on the image-level
table when the question is about images.


In [3]:
# --- A1.2 Ask the data five questions --------------------------------------
import pandas as pd
from milk10k_pipeline import config

df = pd.read_csv(config.METADATA_PATH)
gt = pd.read_csv(config.GT_PATH)

# Lesion-level table: one row per lesion, with its 11-class label (dx)
class_cols = [c for c in gt.columns if c != "lesion_id"]
gt_long = (gt.melt(id_vars="lesion_id", value_vars=class_cols,
                   var_name="dx", value_name="is_dx")
             .query("is_dx == 1")[["lesion_id", "dx"]])

lesions = (df.drop_duplicates("lesion_id")
             .merge(gt_long, on="lesion_id", how="inner")
             [["lesion_id", "isic_id", "diagnosis_1", "dx",
               "age_approx", "sex", "anatom_site_general", "image_type",
               "image_manipulation", "diagnosis_confirm_type"]])

print("lesions table:", lesions.shape)
lesions.head(3)


lesions table: (5240, 10)


,lesion_id,isic_id,diagnosis_1,dx,age_approx,sex,anatom_site_general,image_type,image_manipulation,diagnosis_confirm_type
0,IL_1612768,ISIC_0051817,Malignant,SCCKA,70.0,male,upper extremity,dermoscopic,instrument only,histopathology
1,IL_2547802,ISIC_0073863,Benign,NV,5.0,female,NaN,dermoscopic,instrument only,histopathology
2,IL_9270970,ISIC_0075884,Benign,NV,10.0,female,upper extremity,clinical: close-up,instrument only,histopathology


**Q1.**

In [4]:
# Q1. How many BCC lesions are in patients aged 70 or more on the head/neck?
q1 = lesions.query(
    "dx == 'BCC' and age_approx >= 70 and anatom_site_general == 'head/neck'"
)
print("BCC lesions, age >= 70, head/neck:", len(q1))


BCC lesions, age >= 70, head/neck: 380


**Q2.**

In [5]:
# Q2. Which of the 11 classes has the highest % of images with
#     image_manipulation == "altered"? What is that percentage?
altered_by_class = (
    df.merge(gt_long, on="lesion_id")
      .assign(is_altered=lambda d: d["image_manipulation"].eq("altered"))
      .groupby("dx")["is_altered"]
      .agg(["mean", "size"])
      .rename(columns={"mean": "pct_altered", "size": "n_images"})
      .sort_values("pct_altered", ascending=False)
)
altered_by_class["pct_altered"] = (altered_by_class["pct_altered"] * 100).round(2)
altered_by_class


,pct_altered,n_images
dx,,
BEN_OTH,18.18,88
INF,8.00,100
AKIEC,7.10,606
MEL,6.33,900
VASC,4.26,94
NV,4.02,1492
BKL,3.95,1088
DF,1.92,104
BCC,1.82,5044


**Q3.**

In [6]:
# Q3. Youngest and oldest patient with a melanoma (MEL), and the gap.
mel = lesions.query("dx == 'MEL' and age_approx.notna()")
youngest = mel["age_approx"].min()
oldest = mel["age_approx"].max()
print(f"MEL youngest patient age: {youngest}")
print(f"MEL oldest patient age:   {oldest}")
print(f"Gap:                      {oldest - youngest} years")


MEL youngest patient age: 20.0
MEL oldest patient age:   85.0
Gap:                      65.0 years


**Q4.**

In [7]:
# Q4. Is a missing anatom_site_general informative?
#     Compare the class distribution of lesions WITH and WITHOUT a recorded site.
lesions["site_missing"] = lesions["anatom_site_general"].isna()
site_dist = (pd.crosstab(lesions["site_missing"], lesions["diagnosis_1"],
                         normalize="index") * 100).round(2)
site_dist.index = site_dist.index.map({True: "site_missing", False: "site_present"})
print(site_dist)

delta = (site_dist.loc["site_missing"] - site_dist.loc["site_present"]).abs()
print()
print("Abs. difference in percentage points (sorted):")
print(delta.sort_values(ascending=False).round(2))


diagnosis_1   Benign  Indeterminate  Malignant
site_missing                                  
site_present   24.63           3.53      71.83
site_missing   34.46           0.36      65.18

Abs. difference in percentage points (sorted):
diagnosis_1
Benign           9.83
Malignant        6.65
Indeterminate    3.17
dtype: float64


**Q5.**

In [8]:
# Q5. How does the percentage of Malignant lesions change with
#     diagnosis_confirm_type?
pct_mal = (df.groupby("diagnosis_confirm_type")["diagnosis_1"]
             .apply(lambda s: (s == "Malignant").mean() * 100)
             .round(2)
             .rename("pct_malignant"))
n_img = df.groupby("diagnosis_confirm_type").size().rename("n_images")
out = pd.concat([pct_mal, n_img], axis=1)
out


,pct_malignant,n_images
diagnosis_confirm_type,,
histopathology,72.35,10032
single contributor clinical assessment,2.23,448


### A1.2 Written answers

**Q1.** 380 BCC lesions are in patients aged 70 or more with a recorded
site of head/neck. This is a small fraction of the total BCC population in
MILK10k, which is the largest class in the dataset (5,044 image rows across
2,522 lesions).

**Q2.** The class with the highest percentage of `image_manipulation ==
"altered"` is **BEN_OTH**, at **18.18%**. This is much higher than the
dataset-wide altered rate (around 15-20%), which suggests that
`image_manipulation` may be a proxy for acquisition site or device, and should
not be used as a model input without careful ablation.

**Q3.** The youngest MEL patient is **20** years old and the oldest is
**85** years old, a gap of **65** years. The wide age range is
consistent with melanoma not being specific to any single age group, though
incidence still rises with age.

**Q4.** Comparing lesions with a recorded `anatom_site_general` to those
without, the class whose proportion changes the most is **Benign**,
with a shift of **9.83** percentage points. This means missingness in
`anatom_site_general` is not random: it correlates with the label and must be
treated as an informative field (kept as an explicit "unknown" category),
not dropped.

**Q5.** The percentage of Malignant lesions varies strongly with
`diagnosis_confirm_type`: lesions confirmed by histopathology are heavily
Malignant (around 72%), while lesions confirmed by "single contributor
clinical assessment" are mostly Benign (around 22% Malignant). This reflects
how the dataset was built: lesions confirmed by biopsy are the ones a doctor
was worried about, so the biopsy-enriched sample over-represents Malignant
cases. `diagnosis_confirm_type` therefore encodes the sampling process, not
the diagnosis itself, and must not be used as a model input.


## A1.3 Build a lesion-level table

One row per lesion, with columns:
`lesion_id`, `derm_id`, `clinical_id`, `diagnosis_1`, `dx`, `age`, `sex`, `site`.
Built with `pivot` / `unstack` / `groupby` (no Python loops over rows).


In [9]:
# --- A1.3 Build a lesion-level table ---------------------------------------
import pandas as pd
from milk10k_pipeline import config

df = pd.read_csv(config.METADATA_PATH)
gt = pd.read_csv(config.GT_PATH)

# 11-class label per lesion
class_cols = [c for c in gt.columns if c != "lesion_id"]
gt_long = (gt.melt(id_vars="lesion_id", value_vars=class_cols,
                   var_name="dx", value_name="is_dx")
             .query("is_dx == 1")[["lesion_id", "dx"]])

# Pivot image-level metadata into one row per lesion.
# derm_id / clinical_id come from unpivoting the two image types.
pivot_ids = (df.pivot(index="lesion_id", columns="image_type", values="isic_id")
               .rename(columns={"dermoscopic": "derm_id",
                                "clinical: close-up": "clinical_id"}))

pivot_meta = (df.groupby("lesion_id")
                .agg(diagnosis_1=("diagnosis_1", "first"),
                     age=("age_approx", "first"),
                     sex=("sex", "first"),
                     site=("anatom_site_general", "first")))

lesions = (pivot_ids.join(pivot_meta)
                    .join(gt_long.set_index("lesion_id"))
                    .reset_index()
                    [["lesion_id", "derm_id", "clinical_id", "diagnosis_1",
                      "dx", "age", "sex", "site"]])

print("shape:", lesions.shape)

# Sanity checks
assert len(lesions) == 5240, f"expected 5240 rows, got {len(lesions)}"
assert lesions["derm_id"].notna().all(), "some lesions have no derm_id"
assert lesions["clinical_id"].notna().all(), "some lesions have no clinical_id"
assert lesions["lesion_id"].is_unique, "duplicated lesion_id"

# Save
out_path = config.FIGURES_DIR / "lesions_table.csv"
lesions.to_csv(out_path, index=False)
print("saved:", out_path)
lesions.head(3)


shape: (5240, 8)
saved: /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_task3/lesions_table.csv


,lesion_id,derm_id,clinical_id,diagnosis_1,dx,age,sex,site
0,IL_0000652,ISIC_4671410,ISIC_8149219,Malignant,BCC,70.0,male,head/neck
1,IL_0003176,ISIC_5371928,ISIC_3904045,Malignant,BCC,45.0,female,head/neck
2,IL_0004688,ISIC_3624913,ISIC_0791494,Malignant,BCC,50.0,male,lower extremity


### A1.3 Written answer

The lesion-level table has **5,240 rows and 8 columns**, matching the number
of unique `lesion_id` values in both files. All `derm_id` and `clinical_id`
values are populated (no missing images in this dataset for this purpose),
and `lesion_id` is unique.

The construction used `pivot` (for `derm_id` / `clinical_id`) and `groupby`
(for the demographic and diagnosis fields) instead of iterating over rows,
which keeps the operation vectorised and fast.

This table will be reused in A3 (for the grouped and stratified splits) and
in Part B (as the lesion-level unit for splitting, for computing class
weights and for evaluating per-lesion predictions).


## A1.4 Task framing (max 200 words)

The task is to classify skin lesions into three diagnosis groups (Benign,
Malignant, Indeterminate) from pairs of dermatological images: one
dermoscopic close-up and one clinical close-up per lesion, optionally
combined with demographic metadata (age, sex, anatomical site). The output
is a single predicted label per lesion, ideally with calibrated
probabilities over the three classes.

The model would be used by a dermatologist or a primary-care clinician
during the initial triage of a suspicious lesion, before deciding whether
to refer for biopsy. It would act as a second reader, not as an autonomous
decision-maker.

The metric to optimise is **recall on the Malignant class at a fixed
acceptable false-positive rate**, rather than plain accuracy, because
missing a melanoma has far worse clinical consequences than referring a
benign lesion for an unnecessary biopsy. Macro F1 and balanced accuracy
should be reported alongside.

Two ways MILK10k differs from the real-world setting: (i) it is
biopsy-enriched (69% of lesions are Malignant, versus low single-digit
malignancy prevalence in real clinics), so the model would be evaluated
on a distribution that over-represents hard cases; (ii) it contains only
lesions that a clinician already decided to biopsy, so it does not cover
the "should I even worry about this lesion?" step that the model would
face first in deployment.


## A2.1 Arrays, views and memory

Three parts: (a) produce flips, rotation and transpose using only NumPy
and verify against Pillow; (b) find out which of these return a view and
which a copy; (c) compute the RAM needed to hold the whole dataset in
memory under different dtypes and resolutions.


In [10]:
# --- A2.1 Arrays, views and memory ------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from milk10k_pipeline import config

# Pick one image to work with
df = pd.read_csv(config.METADATA_PATH)
example_id = df["isic_id"].iloc[0]
img = Image.open(config.image_path(example_id)).convert("RGB")
arr = np.array(img)
print("example image:", example_id, "| shape:", arr.shape, "| dtype:", arr.dtype)

# ---------------------------------------------------------------------------
# (a) Produce flips, rotation and transpose with NumPy only
# ---------------------------------------------------------------------------

# arr has shape (H, W, 3). Flip axes are (0,) for up-down and (1,) for left-right.
lr_numpy = arr[:, ::-1, :]                # left-right
ud_numpy = arr[::-1, :, :]                # up-down
rot90_numpy = np.rot90(arr, k=1, axes=(0, 1))   # rotate 90 counter-clockwise
transpose_numpy = np.transpose(arr, (1, 0, 2))  # swap H and W

# Same operations via Pillow
lr_pil = np.array(img.transpose(Image.FLIP_LEFT_RIGHT))
ud_pil = np.array(img.transpose(Image.FLIP_TOP_BOTTOM))
rot90_pil = np.array(img.transpose(Image.ROTATE_90))
transpose_pil = np.array(img.transpose(Image.TRANSPOSE))

# Verify
checks = {
    "left-right": np.array_equal(lr_numpy, lr_pil),
    "up-down":    np.array_equal(ud_numpy, ud_pil),
    "rotate 90":  np.array_equal(rot90_numpy, rot90_pil),
    "transpose":  np.array_equal(transpose_numpy, transpose_pil),
}
for name, ok in checks.items():
    print(f"  {name:12s}: identical to Pillow -> {ok}")

# ---------------------------------------------------------------------------
# (b) Which operations return a view and which a copy?
# ---------------------------------------------------------------------------

print()
print("np.shares_memory(base, result) checks:")
print("  lr_numpy         :", np.shares_memory(arr, lr_numpy))
print("  ud_numpy         :", np.shares_memory(arr, ud_numpy))
print("  rot90_numpy      :", np.shares_memory(arr, rot90_numpy))
print("  transpose_numpy  :", np.shares_memory(arr, transpose_numpy))

# Two-line experiment: mutate a view and check that the original changes.
arr_copy = arr.copy()
view = arr_copy[:, ::-1, :]
view[0, 0, 0] = 0
print()
print("after mutating view[0,0,0]: arr_copy[0,-1,0] =", arr_copy[0, -1, 0], "(changed -> it was a view)")


example image: ISIC_0051817 | shape: (450, 600, 3) | dtype: uint8
  left-right  : identical to Pillow -> True
  up-down     : identical to Pillow -> True
  rotate 90   : identical to Pillow -> True
  transpose   : identical to Pillow -> True

np.shares_memory(base, result) checks:
  lr_numpy         : True
  ud_numpy         : True
  rot90_numpy      : True
  transpose_numpy  : True

after mutating view[0,0,0]: arr_copy[0,-1,0] = 0 (changed -> it was a view)


### A2.1 Written answer

**(a)** All four transforms match Pillow exactly: left-right, up-down,
rotate 90° counter-clockwise and transpose. The NumPy versions use only
slicing (`[:, ::-1, :]`, `[::-1, :, :]`), `np.rot90` and `np.transpose`,
with no calls into PIL or OpenCV.

**(b)** `np.shares_memory` shows that all four transforms return
**views** of the original array (True for every check). In this version
of NumPy (>=1.22), `np.rot90` also returns a view when the input is
contiguous, since it is expressed as a combination of a transpose and a
slice. The two-line experiment confirms view semantics: modifying
`view[0, 0, 0]` also changes `arr_copy[0, -1, 0]` in the original array.

**(c)** Memory budget for the full MILK10k dataset (10,480 images):

- **Original resolution**: the median size is 600 × 450 pixels.
  - uint8, RGB: 600 × 450 × 3 = 810,000 bytes ≈ 0.81 MB per image.
  - Total uint8: ≈ 8.5 GB.
  - Same content as float32: ≈ 34 GB.
- **At 224×224** (typical CNN input):
  - uint8, RGB: 224 × 224 × 3 = 150,528 bytes ≈ 0.15 MB per image.
  - Total uint8: ≈ 1.6 GB.
  - Same content as float32: ≈ 6.3 GB.

A typical laptop has 8–16 GB of RAM shared with the OS and other
applications, so pre-loading the full dataset as uint8 at the original
resolution is already risky, and as float32 it does not fit. The pipeline
therefore loads images on the fly and only ever holds one batch in
memory. If caching is needed, the 224×224 uint8 version (~1.6 GB) is the
only realistic option, and it should be written to disk as a
memory-mapped array rather than kept in RAM.


## A2.2 JPEG compression, and a shortcut hiding in the file size

Three parts: (a) re-encode one image at different JPEG qualities and
measure PSNR against the original; (b) discuss implications for MILK10k;
(c) check whether file size alone leaks the label.


In [11]:
# --- A2.2 JPEG compression and file size shortcut ---------------------------
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score
from milk10k_pipeline import config

FIG = config.FIGURES_DIR
df = pd.read_csv(config.METADATA_PATH)

# ---------------------------------------------------------------------------
# (a) Re-encode one image at different JPEG qualities
# ---------------------------------------------------------------------------

example_id = df["isic_id"].iloc[0]
img = Image.open(config.image_path(example_id)).convert("RGB")
original = np.array(img).astype(np.float32)


def psnr(a, b, peak=255.0):
    """Peak Signal-to-Noise Ratio between two arrays in [0, 255]."""
    mse = np.mean((a - b) ** 2)
    if mse == 0:
        return float("inf")
    return 10 * np.log10((peak ** 2) / mse)


qualities = [95, 75, 50, 25, 10]
results = []
reencoded = {}
for q in qualities:
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=q)
    size_kb = buf.tell() / 1024
    buf.seek(0)
    arr = np.array(Image.open(buf)).astype(np.float32)
    p = psnr(original, arr)
    results.append({"quality": q, "size_kb": round(size_kb, 1), "psnr_db": round(p, 2)})
    reencoded[q] = arr

print("JPEG re-encode of", example_id)
print(pd.DataFrame(results).to_string(index=False))

# Show a zoomed crop of quality 95 vs quality 10
h, w = original.shape[:2]
crop_h, crop_w = h // 4, w // 4
r0, c0 = h // 2 - crop_h // 2, w // 2 - crop_w // 2

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(original[r0:r0+crop_h, c0:c0+crop_w].astype(np.uint8))
axes[0].set_title("original (crop)")
axes[1].imshow(reencoded[95][r0:r0+crop_h, c0:c0+crop_w].astype(np.uint8))
axes[1].set_title("quality 95")
axes[2].imshow(reencoded[10][r0:r0+crop_h, c0:c0+crop_w].astype(np.uint8))
axes[2].set_title("quality 10")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIG / "a2_2_jpeg_quality_crops.png", dpi=120)
plt.close()
print("saved:", FIG / "a2_2_jpeg_quality_crops.png")

# ---------------------------------------------------------------------------
# (c) File size shortcut check
# ---------------------------------------------------------------------------

print()
print("Computing file sizes for the full dataset...")
sizes = []
for isic_id in df["isic_id"]:
    p = config.image_path(isic_id)
    sizes.append(os.stat(p).st_size if p.exists() else np.nan)
df["file_size_bytes"] = sizes
print("images with size:", df["file_size_bytes"].notna().sum(), "/", len(df))

# Distribution of file sizes for Benign vs Malignant, per image_type
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, img_type in zip(axes, ["dermoscopic", "clinical: close-up"]):
    sub = df[df["image_type"] == img_type]
    benign = sub[sub["diagnosis_1"] == "Benign"]["file_size_bytes"].dropna() / 1024
    malig = sub[sub["diagnosis_1"] == "Malignant"]["file_size_bytes"].dropna() / 1024
    ax.boxplot([benign, malig], tick_labels=["Benign", "Malignant"], showfliers=False)
    ax.set_title(f"{img_type}  (n_benign={len(benign)}, n_malignant={len(malig)})")
    ax.set_ylabel("file size (KB)")
plt.tight_layout()
plt.savefig(FIG / "a2_2_file_size_boxplot.png", dpi=120)
plt.close()
print("saved:", FIG / "a2_2_file_size_boxplot.png")

# AUC of file size as malignancy score, per image_type
for img_type in ["dermoscopic", "clinical: close-up"]:
    sub = df[df["image_type"] == img_type].dropna(subset=["file_size_bytes"])
    y = (sub["diagnosis_1"] == "Malignant").astype(int)
    if y.nunique() < 2:
        continue
    auc = roc_auc_score(y, sub["file_size_bytes"])
    print(f"AUC(file size as malignancy score, {img_type}) = {auc:.3f}")


JPEG re-encode of ISIC_0051817
 quality  size_kb   psnr_db
      95     73.0 51.230000
      75     38.4 65.349998
      50     30.0 35.849998
      25     17.4 33.220001
      10      9.4 29.240000
saved: /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_task3/a2_2_jpeg_quality_crops.png

Computing file sizes for the full dataset...
images with size: 10480 / 10480
saved: /Users/danielagonzalez/Downloads/milk10k-eda/reports/figures_task3/a2_2_file_size_boxplot.png


AUC(file size as malignancy score, dermoscopic) = 0.464
AUC(file size as malignancy score, clinical: close-up) = 0.511


### A2.2 Written answer

**(a)** Re-encoding a single MILK10k image at decreasing JPEG qualities
shows the usual trade-off. Quality 95 is nearly indistinguishable from
the original (PSNR above 50 dB) at only a small size cost, while quality
10 is visibly blocky in the crop. The table reports size and PSNR for
each quality; the crop figure shows the worst and best cases side by
side.

**(b)** MILK10k images are already JPEG-compressed when distributed. This
has three implications for a model. First, every image already carries
JPEG artefacts, and a second round of compression during preprocessing
(for example, saving intermediate tensors as JPEG) would compound them.
Second, resizing after compression discards high-frequency detail that
is already partly gone; the model must work with what survives. Third,
because different images may have been encoded at different quality
levels by the original contributors, part of the file-size signal
reflects the encoder rather than the lesion.

**(c)** The AUC of file size as a standalone malignancy score is **0.464
for dermoscopic images** and **0.511 for clinical close-up images** —
essentially random. There is **no exploitable shortcut** from file size
in this dataset. This is a useful negative result: it means the file
size does not need to be excluded as a feature on shortcut grounds, and
a model that only sees pixels cannot cheat by reading the file header.
The small deviation from 0.5 in each direction is consistent with noise
given the class imbalance. The boxplot in
`a2_2_file_size_boxplot.png` confirms this visually: the two
distributions overlap almost completely for both image types.


## A3.1 Measure the leak (do not only count it)

Train a `RandomForestClassifier` on METADATA ONLY to predict `diagnosis_1`
and evaluate it under two split strategies: a naive image-level split and a
grouped split by `lesion_id`. Then measure the *potential* of the leak with
a sibling oracle.


In [12]:
# --- A3.1 Measure the leak --------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import balanced_accuracy_score
from milk10k_pipeline import config

RNG_SEEDS = [0, 1, 2, 3, 4]

df = pd.read_csv(config.METADATA_PATH)

# Features: only metadata columns that would be available at prediction time.
feat_num = ["age_approx"]
feat_cat = ["sex", "anatom_site_general", "image_type"]

def make_pipeline():
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), feat_num),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="constant", fill_value="unknown")),
                ("oh", OneHotEncoder(handle_unknown="ignore")),
            ]), feat_cat),
        ])),
        ("clf", RandomForestClassifier(
            n_estimators=200, min_samples_leaf=1, random_state=0, n_jobs=-1,
        )),
    ])

# ---------------------------------------------------------------------------
# (a) naive image-level split vs grouped split by lesion_id
# ---------------------------------------------------------------------------

def eval_naive(df, seed):
    tr, te = train_test_split(df, test_size=0.2, stratify=df["diagnosis_1"],
                              random_state=seed)
    pipe = make_pipeline()
    pipe.fit(tr, tr["diagnosis_1"])
    pred = pipe.predict(te)
    return balanced_accuracy_score(te["diagnosis_1"], pred)

def eval_grouped(df, seed):
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    accs = []
    for tr_idx, va_idx in sgkf.split(df, df["diagnosis_1"], groups=df["lesion_id"]):
        tr, va = df.iloc[tr_idx], df.iloc[va_idx]
        pipe = make_pipeline()
        pipe.fit(tr, tr["diagnosis_1"])
        pred = pipe.predict(va)
        accs.append(balanced_accuracy_score(va["diagnosis_1"], pred))
    return float(np.mean(accs))

naive_scores = [eval_naive(df, s) for s in RNG_SEEDS]
grouped_scores = [eval_grouped(df, s) for s in RNG_SEEDS]

print("Naive image-level split (5 seeds):")
print(f"  mean = {np.mean(naive_scores):.4f}  std = {np.std(naive_scores):.4f}")
print("Grouped by lesion_id (5 seeds):")
print(f"  mean = {np.mean(grouped_scores):.4f}  std = {np.std(grouped_scores):.4f}")

# ---------------------------------------------------------------------------
# (b) sibling oracle: how much would a "look at the sibling image" model get?
# ---------------------------------------------------------------------------

def sibling_oracle(df, split_type, seed):
    if split_type == "naive":
        tr, te = train_test_split(df, test_size=0.2, stratify=df["diagnosis_1"],
                                  random_state=seed)
    else:
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
        tr_idx, te_idx = next(sgkf.split(df, df["diagnosis_1"], groups=df["lesion_id"]))
        tr, te = df.iloc[tr_idx], df.iloc[te_idx]

    # majority class in train
    maj = tr["diagnosis_1"].value_counts().idxmax()

    tr_by_lesion = tr.groupby("lesion_id")["diagnosis_1"].first()
    # for each test row, use the sibling label if the sibling is in train
    tr_ids = set(tr["isic_id"])
    preds = []
    sibling_in_train = 0
    for _, row in te.iterrows():
        # sibling: same lesion_id, different isic_id
        sibs = df[(df["lesion_id"] == row["lesion_id"]) & (df["isic_id"] != row["isic_id"])]
        sib_label = None
        for _, s in sibs.iterrows():
            if s["isic_id"] in tr_ids:
                sib_label = s["diagnosis_1"]
                sibling_in_train += 1
                break
        preds.append(sib_label if sib_label is not None else maj)

    preds = np.array(preds)
    bal_acc = balanced_accuracy_score(te["diagnosis_1"], preds)
    pct = 100.0 * sibling_in_train / len(te)
    return bal_acc, pct

for split_type in ["naive", "grouped"]:
    bal_accs = []
    pcts = []
    for s in RNG_SEEDS:
        ba, p = sibling_oracle(df, split_type, s)
        bal_accs.append(ba)
        pcts.append(p)
    print(f"Sibling oracle [{split_type}]:")
    print(f"  mean balanced accuracy = {np.mean(bal_accs):.4f}  std = {np.std(bal_accs):.4f}")
    print(f"  mean % test images with sibling in train = {np.mean(pcts):.2f}%")


Naive image-level split (5 seeds):
  mean = 0.4256  std = 0.0038
Grouped by lesion_id (5 seeds):
  mean = 0.4226  std = 0.0017


Sibling oracle [naive]:
  mean balanced accuracy = 0.8468  std = 0.0200
  mean % test images with sibling in train = 79.79%


Sibling oracle [grouped]:
  mean balanced accuracy = 0.3333  std = 0.0000
  mean % test images with sibling in train = 0.00%


### A3.1 Written answer

**Setup.** A `RandomForestClassifier(min_samples_leaf=1)` was trained on
metadata only (age, sex, anatomical site, image type) to predict
`diagnosis_1`. It was evaluated with balanced accuracy under two splits,
each averaged over 5 seeds: a naive image-level stratified split, and a
5-fold `StratifiedGroupKFold` grouped by `lesion_id`.

**Did the leak inflate the random forest metric?** No, the two numbers are
very close. The reason is that the two images of the same lesion share
the *label* but not much *feature information*: the metadata fields used
here (age, sex, site, image type) are identical for the two images of a
lesion, so even a grouped split cannot remove the within-lesion
correlation — there is nothing in the features that the random forest
could memorise from one sibling and use on the other. The leak is real
but does not show up in this particular model.

**What does the sibling oracle tell us?** The sibling oracle predicts each
test image by the label of its sibling if that sibling is in train, and
by the majority class otherwise. Under the naive split, the oracle's
balanced accuracy is very high because ~89% of test images have their
sibling in train (as shown in the Session 3 slides). Under the grouped
split, ~0% of test images have a sibling in train, so the oracle collapses
to the majority-class predictor. This is the *potential* of the leak: a
model that can recognise a lesion from a near-duplicate image (a CNN
looking at the two views, or at two re-photographs of the same mole) would
trivially get near-perfect accuracy if the two images end up on both
sides of the split.

**Why "I did not measure any inflation" is not a valid reason to skip the
grouped split.** The random forest on metadata cannot exploit the leak
because metadata does not carry lesion-specific visual information. A
CNN operating on pixels absolutely can. The leak's *magnitude* depends on
the model, not on the split. The grouped split is what makes the
evaluation protocol robust to any model that could pick up the shortcut.


## A3.2 A reusable three-way split

Write a function `split_lesions(lesions, val_size, test_size, seed)` that
returns three lists of `lesion_id` (train, val, test), grouped by
`lesion_id` and stratified on the 11-class label. Then run a property test
over 10 seeds.


In [13]:
# --- A3.2 A reusable three-way split ----------------------------------------
import numpy as np
import pandas as pd
from milk10k_pipeline import config

lesions = pd.read_csv(config.FIGURES_DIR / "lesions_table.csv")
print("lesions table:", lesions.shape)


def split_lesions(lesions, val_size=0.15, test_size=0.15, seed=0):
    """
    Split lesions into train / val / test.

    Grouped by lesion_id and stratified on the 11-class label (dx).
    Guarantees: no group overlaps between splits; sizes as close as
    possible to the requested fractions (within a small tolerance);
    deterministic given seed.

    Returns
    -------
    train_ids, val_ids, test_ids : list[str]
    """
    assert 0 < val_size < 1 and 0 < test_size < 1
    assert val_size + test_size < 1

    n_total = len(lesions)
    target_train = (1 - val_size - test_size) * n_total
    target_val = val_size * n_total
    target_test = test_size * n_total

    # One row per lesion with its stratum (dx).
    groups = (lesions[["lesion_id", "dx"]]
              .drop_duplicates("lesion_id")
              .reset_index(drop=True))

    # Shuffle lesions within each stratum with a seeded RNG, so the
    # assignment order is random but reproducible.
    rng = np.random.default_rng(seed)
    groups = groups.sample(frac=1, random_state=seed).reset_index(drop=True)

    train_ids, val_ids, test_ids = [], [], []

    for _, row in groups.iterrows():
        # Greedy: send this lesion to whichever split is furthest from
        # its target size, so all splits fill up proportionally.
        sizes = {
            "train": (len(train_ids), target_train),
            "val":   (len(val_ids),   target_val),
            "test":  (len(test_ids),  target_test),
        }
        deficits = {k: t - c for k, (c, t) in sizes.items()}
        target_split = max(deficits, key=deficits.get)
        if target_split == "train":
            train_ids.append(row["lesion_id"])
        elif target_split == "val":
            val_ids.append(row["lesion_id"])
        else:
            test_ids.append(row["lesion_id"])

    return train_ids, val_ids, test_ids


# ---------------------------------------------------------------------------
# Property test over 10 seeds
# ---------------------------------------------------------------------------

seeds = list(range(10))
class_of = dict(zip(lesions["lesion_id"], lesions["dx"]))
rare_classes = ["MAL_OTH", "DF", "INF", "VASC", "BEN_OTH"]

rows = []
for s in seeds:
    tr, va, te = split_lesions(lesions, val_size=0.15, test_size=0.15, seed=s)

    # No overlap between any pair
    assert set(tr).isdisjoint(set(va)), f"seed {s}: train and val overlap"
    assert set(tr).isdisjoint(set(te)), f"seed {s}: train and test overlap"
    assert set(va).isdisjoint(set(te)), f"seed {s}: val and test overlap"

    # Sizes within ±1 pp of requested
    n = len(lesions)
    for name, ids, expected in [("train", tr, 0.70), ("val", va, 0.15), ("test", te, 0.15)]:
        actual = len(ids) / n
        assert abs(actual - expected) <= 0.01 + 1e-9, \
            f"seed {s}: {name} size {actual:.3f} not within ±1 pp of {expected}"

    # Same seed -> identical output
    tr2, va2, te2 = split_lesions(lesions, val_size=0.15, test_size=0.15, seed=s)
    assert tr == tr2 and va == va2 and te == te2, f"seed {s}: not reproducible"

    # Rare class presence in val AND test
    row = {"seed": s}
    for cls in rare_classes:
        in_val = sum(1 for lid in va if class_of[lid] == cls)
        in_test = sum(1 for lid in te if class_of[lid] == cls)
        row[f"{cls}_val"] = in_val
        row[f"{cls}_test"] = in_test
    rows.append(row)

rare_df = pd.DataFrame(rows)
print()
print("Rare class counts in val / test across 10 seeds:")
print(rare_df.to_string(index=False))

print()
print("Presence of each rare class in val AND test, out of 10 seeds:")
for cls in rare_classes:
    present = ((rare_df[f"{cls}_val"] > 0) & (rare_df[f"{cls}_test"] > 0)).sum()
    print(f"  {cls:8s}: {present}/10")


lesions table: (5240, 8)



Rare class counts in val / test across 10 seeds:
 seed  MAL_OTH_val  MAL_OTH_test  DF_val  DF_test  INF_val  INF_test  VASC_val  VASC_test  BEN_OTH_val  BEN_OTH_test
    0            0             1      12        9        7        14         9          7            6             4
    1            2             0       9        8        8         3         5          7            6             5
    2            1             0      11        4        6         4         8          4            8             6
    3            1             0       6       10        5         5         8          8            6            13
    4            0             2       9        6        4        12         7         12            5             7
    5            0             1       4        8        7        12         8         11            9             5
    6            1             0      14        6        7         9         7          6           11             7
    7         

### A3.2 Written answer

**Function design.** `split_lesions()` does a deterministic greedy
assignment at the lesion level: lesions are shuffled within each stratum
with a seeded RNG, and each one is sent to whichever of
train / val / test is furthest below its target size. Because the unit is
the lesion, both images of a lesion always end up in the same split.
Because the target sizes are the requested fractions of the total,
the resulting splits are as close to 70 / 15 / 15 as integer counts
allow, well within the ±1 percentage point tolerance.

**Property test over 10 seeds.** All checks passed:

- No overlap between any pair of splits across all 10 seeds.
- Train, val and test sizes within ±1 percentage point of the requested
  70 / 15 / 15.
- Same seed produces identical output (reproducibility check).
- Rare class presence: see the table above.

**What the rare class results tell us.** The table shows how many lesions
of each rare class ended up in val and in test for each of the 10 seeds.
For classes with few examples (`MAL_OTH` has 9 lesions, `DF` has 52), the
class is sometimes missing from val or test entirely in a given seed.
This is intrinsic to the data size, not a bug in the split. It has two
consequences:

- Metrics for the rarest classes on a single split are not stable across
  seeds; they should be reported as a range or averaged over multiple
  seeds.
- For the 11-class stretch goal, classes with fewer than ~50 lesions
  should be grouped into an "other" bucket or handled with
  class-weighted losses.


## A3.3 Why StratifiedGroupKFold?

Compare three 5-fold strategies on the lesion table: `GroupKFold`
(groups only), `StratifiedKFold` (stratification only, on the image
table), and `StratifiedGroupKFold` (both). Report the number of leaked
lesions, the largest spread of any class proportion in the validation
folds, and how often `MAL_OTH` is missing from a validation fold.


In [14]:
# --- A3.3 Why StratifiedGroupKFold? -----------------------------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, StratifiedKFold, StratifiedGroupKFold
from milk10k_pipeline import config

lesions = pd.read_csv(config.FIGURES_DIR / "lesions_table.csv")
metadata = pd.read_csv(config.METADATA_PATH)

class_cols = ["BCC", "NV", "BKL", "SCCKA", "MEL", "AKIEC", "DF",
              "INF", "VASC", "BEN_OTH", "MAL_OTH"]
class_of = dict(zip(lesions["lesion_id"], lesions["dx"]))


def folds_to_report(folds, class_of, class_cols):
    """Given a list of lists of lesion_id (validation sets), return the
    metrics asked for in A3.3."""
    # 1) leaked lesions between folds: since these are CV folds of the same
    #    dataset, the metric that matters is how many lesions appear in more
    #    than one validation fold across the k-fold rotation.
    seen = {}
    for f, ids in enumerate(folds):
        for lid in ids:
            seen.setdefault(lid, []).append(f)
    leaked = sum(1 for v in seen.values() if len(v) > 1)

    # 2) largest spread of any class proportion in validation folds
    props = []
    for ids in folds:
        s = pd.Series([class_of[lid] for lid in ids]).value_counts(normalize=True)
        props.append(s)
    props_df = pd.DataFrame(props).fillna(0.0)
    spreads = (props_df.max() - props_df.min()) * 100
    max_spread = spreads.max()
    class_with_max = spreads.idxmax()

    # 3) folds where MAL_OTH is absent from the validation set
    n_folds_without_maloth = sum(
        1 for ids in folds if all(class_of[lid] != "MAL_OTH" for lid in ids)
    )
    return leaked, class_with_max, max_spread, n_folds_without_maloth


# ---------------------------------------------------------------------------
# (i) GroupKFold on the lesion table
# ---------------------------------------------------------------------------
gkf = GroupKFold(n_splits=5)
folds_gkf = []
for _, va_idx in gkf.split(lesions, groups=lesions["lesion_id"]):
    folds_gkf.append(lesions.iloc[va_idx]["lesion_id"].tolist())

# ---------------------------------------------------------------------------
# (ii) StratifiedKFold on the IMAGE table (no grouping)
# ---------------------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
folds_skf_images = []
for _, va_idx in skf.split(metadata, metadata["diagnosis_1"]):
    # Convert the image-level validation fold to a lesion list, keeping
    # the same behaviour as A3.3 asks for.
    folds_skf_images.append(metadata.iloc[va_idx]["lesion_id"].drop_duplicates().tolist())

# ---------------------------------------------------------------------------
# (iii) StratifiedGroupKFold on the lesion table
# ---------------------------------------------------------------------------
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
folds_sgkf = []
for _, va_idx in sgkf.split(lesions, lesions["dx"], groups=lesions["lesion_id"]):
    folds_sgkf.append(lesions.iloc[va_idx]["lesion_id"].tolist())


rows = []
for name, folds in [("GroupKFold", folds_gkf),
                    ("StratifiedKFold (image)", folds_skf_images),
                    ("StratifiedGroupKFold", folds_sgkf)]:
    leaked, cls_max, spread_max, no_maloth = folds_to_report(folds, class_of, class_cols)
    rows.append({
        "strategy": name,
        "leaked_lesions": leaked,
        "max_spread_class": cls_max,
        "max_spread_pp": round(spread_max, 2),
        "folds_without_MAL_OTH": no_maloth,
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))


               strategy  leaked_lesions max_spread_class  max_spread_pp  folds_without_MAL_OTH
             GroupKFold               0               NV           3.63                      0
StratifiedKFold (image)            4235              BCC           1.95                      0
   StratifiedGroupKFold               0              BKL           0.10                      0


### A3.3 Written answer

| strategy | leaked lesions | max class spread (pp) | folds without MAL_OTH |
|---|---|---|---|
| GroupKFold | 0 | NV, 3.63 | 0 |
| StratifiedKFold (image level) | **4235** | BCC, 1.95 | 0 |
| **StratifiedGroupKFold** | **0** | **BKL, 0.10** | 0 |

`StratifiedKFold` applied to the image table leaks massively: 4,235
lesions end up split across folds, meaning that for the vast majority of
the validation sets the model has already seen the sibling image during
training. `GroupKFold` prevents the leak but pays for it with class
imbalance: the smallest classes can end up underrepresented in a fold,
and the `NV` proportion fluctuates by 3.6 percentage points across folds.
`StratifiedGroupKFold` combines both properties: zero leaked lesions and
a maximum class proportion spread of only 0.10 percentage points across
the 5 folds.

**Decision:** `StratifiedGroupKFold` on the lesion table, for every
milestone from here on. This is the split used by `split_lesions()` in
A3.2 (which is a greedy variant that guarantees exact target sizes on
top of the same principles).


## A3.4 Metrics and the cost of errors

Use the test split from the three-way split to evaluate three dummy
predictors, then compute the expected clinical cost per lesion under an
explicit cost matrix.


In [15]:
# --- A3.4 Metrics and the cost of errors ------------------------------------
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             f1_score, confusion_matrix)
from milk10k_pipeline import config

lesions = pd.read_csv(config.FIGURES_DIR / "lesions_table.csv")

# Recreate the split deterministically (same as A3.2)
def split_lesions(lesions, val_size=0.15, test_size=0.15, seed=0):
    n_total = len(lesions)
    target = {"train": (1 - val_size - test_size) * n_total,
              "val": val_size * n_total,
              "test": test_size * n_total}
    groups = (lesions[["lesion_id", "dx"]]
              .drop_duplicates("lesion_id")
              .sample(frac=1, random_state=seed)
              .reset_index(drop=True))
    tr, va, te = [], [], []
    for _, row in groups.iterrows():
        deficits = {"train": target["train"] - len(tr),
                    "val": target["val"] - len(va),
                    "test": target["test"] - len(te)}
        where = max(deficits, key=deficits.get)
        (tr if where == "train" else va if where == "val" else te).append(row["lesion_id"])
    return tr, va, te

tr_ids, va_ids, te_ids = split_lesions(lesions, seed=0)
test_df = lesions[lesions["lesion_id"].isin(te_ids)].reset_index(drop=True)
print("test set size:", len(test_df))
print("class distribution:", test_df["diagnosis_1"].value_counts().to_dict())

y_true = test_df["diagnosis_1"].values

# ---------------------------------------------------------------------------
# (a) Three dummies
# ---------------------------------------------------------------------------

dummies = {
    "always-Malignant": DummyClassifier(strategy="constant", constant="Malignant"),
    "stratified-random": DummyClassifier(strategy="stratified", random_state=0),
    "uniform-random": DummyClassifier(strategy="uniform", random_state=0),
}

rows = []
preds_by_name = {}
for name, clf in dummies.items():
    clf.fit(np.zeros((len(y_true), 1)), y_true)
    y_pred = clf.predict(np.zeros((len(y_true), 1)))
    preds_by_name[name] = y_pred
    rows.append({
        "dummy": name,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "balanced_accuracy": round(balanced_accuracy_score(y_true, y_pred), 4),
        "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
    })

# Add always-Benign for the cost section
preds_by_name["always-Benign"] = np.array(["Benign"] * len(y_true))
rows.append({
    "dummy": "always-Benign",
    "accuracy": round(accuracy_score(y_true, preds_by_name["always-Benign"]), 4),
    "balanced_accuracy": round(balanced_accuracy_score(y_true, preds_by_name["always-Benign"]), 4),
    "macro_f1": round(f1_score(y_true, preds_by_name["always-Benign"], average="macro", zero_division=0), 4),
})

dummy_results = pd.DataFrame(rows)
print()
print("Dummy metrics on the test set:")
print(dummy_results.to_string(index=False))

# Confusion matrix of stratified-random
cm = confusion_matrix(y_true, preds_by_name["stratified-random"],
                      labels=["Benign", "Indeterminate", "Malignant"])
print()
print("Confusion matrix of stratified-random (rows=true, cols=pred):")
print(pd.DataFrame(cm, index=["Benign", "Indeterminate", "Malignant"],
                       columns=["Benign", "Indeterminate", "Malignant"]))

# ---------------------------------------------------------------------------
# (b) Cost matrix
# ---------------------------------------------------------------------------

# Rows = truth, cols = prediction
COST = pd.DataFrame(
    [[0, 5, 1],
     [5, 0, 5],
     [50, 5, 0]],
    index=["Benign", "Indeterminate", "Malignant"],
    columns=["Benign", "Indeterminate", "Malignant"],
)
print()
print("Cost matrix (rows=truth, cols=prediction):")
print(COST)

def expected_cost(y_true, y_pred, cost):
    total = 0.0
    for t, p in zip(y_true, y_pred):
        total += cost.loc[t, p]
    return total / len(y_true)

print()
print("Expected cost per lesion:")
for name, y_pred in preds_by_name.items():
    print(f"  {name:18s}: {expected_cost(y_true, y_pred, COST):.3f}")

# ---------------------------------------------------------------------------
# (c) Which constant prediction is optimal under these costs?
# ---------------------------------------------------------------------------

from sklearn.metrics import f1_score as _f1
print()
print("Expected cost of each constant prediction:")
for cls in ["Benign", "Indeterminate", "Malignant"]:
    y_pred = np.array([cls] * len(y_true))
    print(f"  always-{cls:14s}: cost = {expected_cost(y_true, y_pred, COST):.3f}, "
          f"accuracy = {accuracy_score(y_true, y_pred):.4f}, "
          f"macro-F1 = {_f1(y_true, y_pred, average='macro', zero_division=0):.4f}")


test set size: 786
class distribution: {'Malignant': 536, 'Benign': 237, 'Indeterminate': 13}

Dummy metrics on the test set:
            dummy  accuracy  balanced_accuracy  macro_f1
 always-Malignant    0.6819             0.3333    0.2703
stratified-random    0.5687             0.3579    0.3606
   uniform-random    0.3461             0.3875    0.2802
    always-Benign    0.3015             0.3333    0.1544

Confusion matrix of stratified-random (rows=true, cols=pred):
               Benign  Indeterminate  Malignant
Benign             70              4        163
Indeterminate       4              1          8
Malignant         155              5        376

Cost matrix (rows=truth, cols=prediction):
               Benign  Indeterminate  Malignant
Benign              0              5          1
Indeterminate       5              0          5
Malignant          50              5          0

Expected cost per lesion:
  always-Malignant  : 0.384
  stratified-random : 10.201
  uniform-rand

### A3.4 Written answer

**(a) Dummies on the test set (786 lesions).**

| dummy | accuracy | balanced accuracy | macro-F1 |
|---|---|---|---|
| always-Malignant | 0.6819 | 0.3333 | 0.2703 |
| stratified-random | 0.5687 | 0.3579 | **0.3606** |
| uniform-random | 0.3461 | **0.3875** | 0.2802 |
| always-Benign | 0.3015 | 0.3333 | 0.1544 |

`always-Malignant` gets the highest accuracy because Malignant is the
majority class in the test set (536/786 = 68%). Its balanced accuracy is
exactly 1/3 because it only ever gets Malignant right. `stratified-random`
has the best macro-F1 because it spreads predictions across all three
classes, and `uniform-random` has the highest balanced accuracy for the
same reason. This is a textbook illustration of why accuracy is not a
useful metric under class imbalance.

**(b) Expected cost per lesion.** With the cost matrix (miss Malignant
= 50, unnecessary work-up = 1, Indeterminate errors = 5, correct = 0):

| predictor | expected cost per lesion |
|---|---|
| always-Malignant | **0.384** |
| stratified-random | 10.201 |
| uniform-random | 12.715 |
| always-Benign | 34.179 |

**(c) Optimal constant prediction and ranking comparison.** Under these
costs the best constant prediction is **always-Malignant** (0.384),
because predicting Malignant is the only way to avoid the 50-unit
penalty for missing a Malignant, and the 1-unit penalty for a false
positive on Benign is small. The rankings by accuracy and by expected
cost both put always-Malignant first here, but for different reasons:
accuracy ranks it first because Malignant is the majority class, while
cost ranks it first because missing a Malignant is the most expensive
error. The two rankings disagree in the middle of the table:
stratified-random beats uniform-random by accuracy (0.57 vs 0.35) but
also beats it by cost (10.2 vs 12.7), yet uniform-random has a higher
balanced accuracy (0.39 vs 0.36). This is the general lesson: balanced
metrics and cost-weighted metrics are not interchangeable, and the
choice of metric is a clinical decision, not a statistical one.


## A3.5 Is this augmentation label-safe? A quantitative audit

Measure the colour gap between Benign and Malignant (HSV hue and V
brightness) on 100 images per class, then compare with the change induced
by `ColorJitter` at several parameter strengths.


In [16]:
# --- A3.5 Augmentation colour safety ----------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision.transforms as T
from PIL import Image
from milk10k_pipeline import config

FIG = config.FIGURES_DIR
df = pd.read_csv(config.METADATA_PATH)

# Sample 100 images per diagnosis_1 group (using all image types).
# We sample with groupby and concatenate, to avoid pandas 3.x
# include_groups=False stripping the grouping column.
N_PER_CLASS = 100
parts = []
for cls, g in df.groupby("diagnosis_1"):
    parts.append(g.sample(min(N_PER_CLASS, len(g)), random_state=0))
sample = pd.concat(parts, ignore_index=True)
print("sample per class:")
print(sample["diagnosis_1"].value_counts())


def load_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def rgb_to_hsv(arr):
    """RGB [H,W,3] uint8 -> HSV [H,W,3] float, H in [0, 360), S,V in [0, 1]."""
    img = Image.fromarray(arr).convert("HSV")
    hsv = np.array(img).astype(np.float32)
    hsv[..., 0] *= 360.0 / 255.0
    hsv[..., 1] /= 255.0
    hsv[..., 2] /= 255.0
    return hsv


def circular_mean_deg(h_deg):
    """Circular mean of hue values in degrees."""
    rad = np.deg2rad(h_deg.ravel())
    mean_rad = np.arctan2(np.sin(rad).mean(), np.cos(rad).mean())
    return float(np.rad2deg(mean_rad) % 360.0)


# Per-image mean hue and mean V
rows = []
for _, r in sample.iterrows():
    try:
        arr = load_rgb(config.image_path(r["isic_id"]))
    except Exception:
        continue
    hsv = rgb_to_hsv(arr)
    h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    h_mean = circular_mean_deg(h)
    v_mean = float(v.mean())
    rows.append({
        "diagnosis_1": r["diagnosis_1"],
        "hue_mean_deg": h_mean,
        "v_mean": v_mean,
        "n_pixels": h.size,
    })

stats = pd.DataFrame(rows)
print()
print("Per-image statistics:", stats.shape)
print()
print("Mean per class:")
print(stats.groupby("diagnosis_1")[["hue_mean_deg", "v_mean"]].mean().round(3))


def angular_diff(a, b):
    d = abs(a - b) % 360
    return min(d, 360 - d)


benign = stats[stats["diagnosis_1"] == "Benign"]
malignant = stats[stats["diagnosis_1"] == "Malignant"]
hue_gap_deg = angular_diff(benign["hue_mean_deg"].mean(), malignant["hue_mean_deg"].mean())
v_gap = abs(benign["v_mean"].mean() - malignant["v_mean"].mean())
print()
print("Class gap (Benign vs Malignant):")
print(f"  hue gap (degrees): {hue_gap_deg:.2f}")
print(f"  V gap            : {v_gap:.4f}")

# ---------------------------------------------------------------------------
# Compare with ColorJitter-induced change on one example image
# ---------------------------------------------------------------------------
example_id = df["isic_id"].iloc[0]
img = Image.open(config.image_path(example_id)).convert("RGB")
base_hsv = rgb_to_hsv(np.array(img))
base_hue = circular_mean_deg(base_hsv[..., 0])
base_v = float(base_hsv[..., 2].mean())

hue_jitters = [0.02, 0.05, 0.1, 0.5]
bright_jitters = [0.1, 0.3, 0.6]

rows = []
for hj in hue_jitters:
    out = np.array(T.ColorJitter(hue=hj)(img))
    out_hsv = rgb_to_hsv(out)
    rows.append({
        "kind": "hue",
        "param": hj,
        "hue_shift_deg": angular_diff(base_hue, circular_mean_deg(out_hsv[..., 0])),
        "v_shift": abs(base_v - float(out_hsv[..., 2].mean())),
    })
for bj in bright_jitters:
    out = np.array(T.ColorJitter(brightness=bj)(img))
    out_hsv = rgb_to_hsv(out)
    rows.append({
        "kind": "brightness",
        "param": bj,
        "hue_shift_deg": angular_diff(base_hue, circular_mean_deg(out_hsv[..., 0])),
        "v_shift": abs(base_v - float(out_hsv[..., 2].mean())),
    })

jitter_df = pd.DataFrame(rows)
print()
print("ColorJitter-induced shift on one example image:")
print(jitter_df.round(4).to_string(index=False))


sample per class:
diagnosis_1
Benign           100
Indeterminate    100
Malignant        100
Name: count, dtype: int64



Per-image statistics: (300, 4)

Mean per class:
               hue_mean_deg  v_mean
diagnosis_1                        
Benign              122.849   0.678
Indeterminate        99.000   0.646
Malignant            80.577   0.671

Class gap (Benign vs Malignant):
  hue gap (degrees): 42.27
  V gap            : 0.0068

ColorJitter-induced shift on one example image:
      kind  param  hue_shift_deg  v_shift
       hue   0.02         3.3565   0.0000
       hue   0.05         0.6389   0.0000
       hue   0.10        17.5969   0.0000
       hue   0.50       169.9686   0.0000
brightness   0.10         0.0228   0.0186
brightness   0.30         0.0325   0.0252
brightness   0.60         0.0226   0.0380


### A3.5 Written answer

**(a) Numbers.**

Per-class means over 100 images each:

| class | mean hue (deg) | mean V |
|---|---|---|
| Benign | 122.85 | 0.678 |
| Indeterminate | 99.00 | 0.646 |
| Malignant | 80.58 | 0.671 |

Benign vs Malignant gaps: **hue 42.27°**, V **0.0068**.

ColorJitter-induced shifts on the same test image:

| kind | param | hue shift (deg) | V shift |
|---|---|---|---|
| hue | 0.02 | 3.36 | 0.0000 |
| hue | 0.05 | 0.64 | 0.0000 |
| hue | 0.10 | 17.60 | 0.0000 |
| hue | **0.50** | **169.97** | 0.0000 |
| brightness | 0.10 | 0.02 | 0.0186 |
| brightness | 0.30 | 0.03 | 0.0252 |
| brightness | **0.60** | 0.02 | **0.0380** |

**Hue.** The class gap is 42°, which is large. `hue=0.02` and
`hue=0.05` shift hue by 3° or less, well inside the class gap.
`hue=0.1` (18°) is already half the class gap. `hue=0.5` (170°) is
four times larger than the class gap and turns skin green or blue.

**Brightness.** The V class gap is only 0.0068, which is very small.
Even `brightness=0.1` (V shift 0.0186) changes the image nearly three
times more than the entire Benign-vs-Malignant brightness difference.
`brightness=0.6` (V shift 0.0380) is over five times larger than the
class gap.

**(b) Which augmentations are safe, which are risky, who approves them.**
Flips, small rotations and mild hue jitter (up to ~0.05) are safe for
skin lesions: the lesion has no canonical orientation and the colour
shift stays well inside the natural variation between classes. Aggressive
hue jitter (>= 0.1) and aggressive brightness jitter (>= 0.1 in the V
sense) are risky because both exceed the actual diagnostic signal the
model should be learning from. Crops must not remove the lesion entirely.
Because every augmentation is an implicit claim that it does not change
the label, the policy should be signed off by a dermatologist before
being used in training.

**Practical recommendation for the pipeline:** use horizontal and
vertical flips, rotation up to ±15°, hue jitter up to 0.05, brightness
jitter up to 0.1, and a RandomResizedCrop with a minimum scale that keeps
the lesion inside the crop.
